In [1]:
## Import necessary packages

%matplotlib tk
# On Macs use osx
# For Windows use qt

import numpy as np
from numpy.random import rand
from landscapeWithOcean import LandscapeWithOcean # Import methods from inside file landscapeWithOcean.py

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm


from matplotlib.animation import FFMpegWriter
metadata = dict(title = "Erosion Model",artist = "Matplotlib")
writer = FFMpegWriter(fps=15,metadata=metadata, bitrate=200000)

In [2]:
def AddHill(Z,NX,NY,xx,yy,r,h):

    for x in range(NX):
        for y in range(NY):
            dx = np.mod(x-xx+NX/2,NX)-NX/2; # difference i-i0 but apply p.b.c. 
            dy = np.mod(y-yy+NY/2,NY)-NY/2;
            dr = np.sqrt(dx**2+dy**2);
            if (dr<r):
                Z[x,y] += h * (np.cos(dr/r*np.pi/2.0))**2;

    return Z

In [3]:
### Define simulation grid and initial conditions

NX = 30*2 #number of rows
NY = 30*2 #number of columns

d  = 5 # grid spacing in meters
dx = d # keep dx=dy for simplicity
dy = d

LX=NX*dx
LY=NY*dy

# small random features in topography to begin erosion
Z = rand(NX,NY)*0.1
for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX
    h  = (0.1+rand())*5
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX/2
    h  = (0.5+rand())*10
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

x = np.arange(NX)
y = np.arange(NY)
X,Y = np.meshgrid(y,x) #strange that y goes first !!!
ZMaxOrg = np.max(Z)
#print(ZMaxOrg)

In [4]:
### Physical Parameters
K = 1.0e-6 # meters^(1-2m)/yr

D = 0.005 # m^2/yr

# uplift rate
# uplift = 0.03 / 600.
uplift = 0.0

In [5]:
### Model parameters

# Time step
dt = d**2 / D / 8. 
#dt = d**2 / D /16. #extra small steps 
print(' dt[years] = ',dt)

#Area exponent A^m, default m=1
m=1

#gradient exponent g^n, default n=1
n=1

#erosion threshold 
theta_c = 10 

 dt[years] =  625.0


In [6]:
# Total simulation time
T = 1000.0 * 625.0

# total number of iterations
n_iter = int(np.round(T/dt))
print('Number of interation: ',n_iter)

Number of interation:  1000


In [7]:
# Initialize landscape 
ls = LandscapeWithOcean(NX,NY)

oceanLevelParameter=0.1  # what does this parameter do?
ls.ComputeOceanVolumeFromOceanLevelParameter(Z,NX,NY,oceanLevelParameter)

ls.pool_check(Z,NX,NY)
ls.A = np.zeros((NX,NY))

Minimum elevation           1.0644956760655715
Maximum elevation           20.229806294341483
Beach level                 2.9810267378931625
Ocean volume                332.81488869039384
Percentage of ocean surface 9.777777777777779


In [8]:
# Set-up figure
def init_figure():
    fig = plt.figure(figsize=(12.,6.))
    plt.show()
    return fig

def update_figure():
        plt.clf()
        ax1 = fig.add_subplot(121,projection='3d')

        # use equal x-y aspect with an explicit vertical exageration
        vert_exag = 4.
        ax1.set_xlim3d(0,max(NX,NY))
        ax1.set_ylim3d(0,max(NX,NY))
        ax1.set_zlim3d(0,ZMaxOrg)

        ax1.set_title('Surface Relief')

#        surf = ax1.plot_surface(X,Y,Z, cmap = cm.terrain, rstride=1, cstride=1,
#                antialiased=False,linewidth=0)

        ZPlot = np.copy(Z)
        ZPlot[ZPlot<ls.ZBeachLevel] = ls.ZBeachLevel 
        ZPlot -= ls.ZBeachLevel
        ax1.plot_surface(X,Y,ZPlot, cmap = cm.terrain, rstride=1, cstride=1,
                antialiased=False,linewidth=0)

        ax2 = fig.add_subplot(122,aspect='equal')
        ax2.set_title('Elevation')

        #im = ax2.pcolor(Z,cmap=cm.terrain)
        im = ax2.pcolor(ZPlot,cmap=cm.coolwarm)
        cs = ax2.contour(ZPlot,6,colors='k')

        # Add a color bar which maps values to colors.
        cbar = fig.colorbar(im, shrink=0.5, aspect=5)
        # Add the contour line levels to the colorbar
        cbar.add_lines(cs)

        #plt.show()
        plt.draw()
        plt.pause(0.0001)

In [9]:
# Set up figure
fig = init_figure()
update_figure()
Znew = np.copy(Z)

for it in range(1,n_iter+1):
    
    ls.calculate_collection_area(Z,NX,NY)
    ls.A *= dx*dy
    
    for i in range(NX):
        iL = np.mod(i-1,NX) # normally i-1 but observe p.b.c.
        iR = np.mod(i+1,NX) # normally i+1 but observe p.b.c.

        for j in range(NY):
            jD = np.mod(j-1,NY) # normally j-1 but observe p.b.c.
            jU = np.mod(j+1,NY) # normally j+1 but observe p.b.c.
  
            if ls.ocean[i,j]>0:
                Psi_z  = 0;
                Phi_z  = 0;
            else:
                if ls.drain[i,j]>0: #this cell is a drain
                    s1 = (Z[iR,j]  - Z[iL,j] )/(2.*dx)
                    s2 = (Z[i,jU]  - Z[i,jD] )/(2.*dy)
                    s3 = (Z[iR,jD] - Z[iL,jU])/(2. * np.sqrt( dx**2 + dy**2) )
                    s4 = (Z[iR,jU] - Z[iL,jD])/(2. * np.sqrt( dx**2 + dy**2) )
                    gradient = (np.sqrt(s1**2 + s2**2) + np.sqrt(s3**2 + s4**2))/2.
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)            

                elif ls.drainage[i,j]>0: #this cell is a drainage point (it drains a pool)
                
                    if (Z[i,j]>=Z[iR,j]) and ls.pool[iR,j]!=ls.drainage[i,j]: 
                        gradient = (Z[i,j]-Z[iR,j])/dx #pool is on my left, I drain to the right, use this gradiant
                    elif (Z[i,j]>=Z[iL,j]) and ls.pool[iL,j]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[iL,j])/dx
                    elif (Z[i,j]>=Z[i,jU]) and ls.pool[i,jU]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jU])/dy
                    elif (Z[i,j]>=Z[i,jD]) and ls.pool[i,jD]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jD])/dy
                    else:
                        gradient = 0.02 # ??? This does happen (maybe when two pools merge)
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)
                
                else: #this cell is a pool, assume it has some mass diffusion but no erosion!
                    Psi_z = 0.
                
                if (Psi_z<0):
                    Psi_z = 0. 

                # diffusion term
                Phi_z = D * ( (Z[iR,j] - 2.*Z[i,j] + Z[iL,j]) / dx**2 \
                            + (Z[i,jU] - 2.*Z[i,j] + Z[i,jD]) / dx**2 )
           
            Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift )*dt  

            dZdt= (Znew[i,j] - Z[i,j]) / dt
            CFL = abs(dZdt) * dt / min(dx,dy)
            if (CFL>1.0):
                print('\nWarning: Time step of',dt,'is probably too large. Safer would be:',dt/CFL)
            
            if (Znew[i,j]<0.):
                Znew[i,j] = 0. # yes, this does happen at the boundary when kept at zero
    
    #Znew[0,:] = 0.0 # resets front boundary to 0
    Z = np.copy(Znew)
    
    ls.pool_check(Z,NX,NY)

    if (np.mod(it,10)==0): 
        print(it,end='')
        update_figure()
        print(' Ocean level=',ls.ZBeachLevel,' Ocean surface fraction=',100*ls.AOcean/(NX*NY));
    else:
        print('.',end='')

update_figure()
print(' Simulation finished.')


.........10 Ocean level= 2.962717934790188  Ocean surface fraction= 11.055555555555555
.........20 Ocean level= 2.953411531501567  Ocean surface fraction= 11.527777777777779
.......30 Ocean level= 2.9527798427436065  Ocean surface fraction= 11.777777777777779
.........40 Ocean level= 2.9517619126390255  Ocean surface fraction= 12.055555555555555
.......50 Ocean level= 2.951001043051485  Ocean surface fraction= 12.25
.........60 Ocean level= 2.9491786839877965  Ocean surface fraction= 12.444444444444445
.......70 Ocean level= 2.9466905440050537  Ocean surface fraction= 12.61111111111111
.......80 Ocean level= 2.9466905440050537  Ocean surface fraction= 12.777777777777779
.......90 Ocean level= 2.946652346852372  Ocean surface fraction= 12.805555555555555
.........100 Ocean level= 2.946652346852372  Ocean surface fraction= 12.944444444444445
.......110 Ocean level= 2.946618720322662  Ocean surface fraction= 13.083333333333334
.........120 Ocean level= 2.946618720322662  Ocean surface fra

In [10]:
#1
#The base surface Z (NX by NZ) is initialized to have small random points that start the erosion simulation process.
#These random points are erosion seeds.
#There are hills that are steep and hills that are gradual that are initialized with loops.
#These hills create a topogrophy that has different features and varies like natural land.

#The oceanLevelParameter variable is a parameter used to calculate the sea level by acting as a scaling factor

#As the model runs, the hills flatten and channels form on the sides of the hills where sediments travel down.
#The model becomes flat and sediments travel to the water.
#The ocean level decreases as this happens in order to keep the ocean volume constant

In [11]:
#2  varied initial conditions

In [12]:
NX = 30*2
NY = 30*2
d = 5
dx = d
dy = d
LX = NX*dx
LY = NY*dy
Z = np.full((NX,NY),1, dtype = np.float64)
center_x = NX/2
center_y = NY/2
r1_center = 15
h1 = 15

for x in range(NX):
    for y in range(NY):
        dr = np.sqrt((x-center_x)**2 + (y-center_y)**2)
        ring1_profile = 20*np.exp(-((dr-r1_center)/2.1)**2)
        Z[x,y] += ring1_profile

Z = AddHill(Z,NX,NY,center_x,center_y,6.4,30.0)

x = np.arange(NX)
y = np.arange(NY)
X,Y = np.meshgrid(y,x)
ZMaxOrg = np.max(Z)

In [13]:
#These altered initial conditions are meant to represent a volcanic island formation.
#I have included a central "volcanoe" formation that is surrounded by a "ring" that represents an older volcanoe formation.
#The volcanoe and ring will start above sea level, but the space between them will be filled by sea.

In [14]:
with writer.saving(fig, "Homework_9_sim1.mp4", dpi=200):
    # Set up figure
    fig = init_figure()
    update_figure()
    Znew = np.copy(Z)

    for it in range(1,n_iter+1):
    
        ls.calculate_collection_area(Z,NX,NY)
        ls.A *= dx*dy
    
        for i in range(NX):
            iL = np.mod(i-1,NX) # normally i-1 but observe p.b.c.
            iR = np.mod(i+1,NX) # normally i+1 but observe p.b.c.

            for j in range(NY):
                jD = np.mod(j-1,NY) # normally j-1 but observe p.b.c.
                jU = np.mod(j+1,NY) # normally j+1 but observe p.b.c.
  
                if ls.ocean[i,j]>0:
                    Psi_z  = 0;
                    Phi_z  = 0;
                else:
                    if ls.drain[i,j]>0: #this cell is a drain
                        s1 = (Z[iR,j]  - Z[iL,j] )/(2.*dx)
                        s2 = (Z[i,jU]  - Z[i,jD] )/(2.*dy)
                        s3 = (Z[iR,jD] - Z[iL,jU])/(2. * np.sqrt( dx**2 + dy**2) )
                        s4 = (Z[iR,jU] - Z[iL,jD])/(2. * np.sqrt( dx**2 + dy**2) )
                        gradient = (np.sqrt(s1**2 + s2**2) + np.sqrt(s3**2 + s4**2))/2.
                        Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)            

                    elif ls.drainage[i,j]>0: #this cell is a drainage point (it drains a pool)
                
                        if (Z[i,j]>=Z[iR,j]) and ls.pool[iR,j]!=ls.drainage[i,j]: 
                            gradient = (Z[i,j]-Z[iR,j])/dx #pool is on my left, I drain to the right, use this gradiant
                        elif (Z[i,j]>=Z[iL,j]) and ls.pool[iL,j]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[iL,j])/dx
                        elif (Z[i,j]>=Z[i,jU]) and ls.pool[i,jU]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jU])/dy
                        elif (Z[i,j]>=Z[i,jD]) and ls.pool[i,jD]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jD])/dy
                        else:
                            gradient = 0.02 # ??? This does happen (maybe when two pools merge)
                        Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)
                
                    else: #this cell is a pool, assume it has some mass diffusion but no erosion!
                        Psi_z = 0.
                
                    if (Psi_z<0):
                        Psi_z = 0. 

                    # diffusion term
                    Phi_z = D * ( (Z[iR,j] - 2.*Z[i,j] + Z[iL,j]) / dx**2 \
                                + (Z[i,jU] - 2.*Z[i,j] + Z[i,jD]) / dx**2 )
           
                Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift )*dt  

                dZdt= (Znew[i,j] - Z[i,j]) / dt
                CFL = abs(dZdt) * dt / min(dx,dy)
                if (CFL>1.0):
                    print('\nWarning: Time step of',dt,'is probably too large. Safer would be:',dt/CFL)
            
                if (Znew[i,j]<0.):
                    Znew[i,j] = 0. # yes, this does happen at the boundary when kept at zero
    
        #Znew[0,:] = 0.0 # resets front boundary to 0
        Z = np.copy(Znew)
    
        ls.pool_check(Z,NX,NY)

        if (np.mod(it,10)==0): 
            print(it,end='')
            update_figure()
            print(' Ocean level=',ls.ZBeachLevel,' Ocean surface fraction=',100*ls.AOcean/(NX*NY));
        else:
            print('.',end='')

    writer.grab_frame()
    update_figure()
    print(' Simulation finished.')


.........10 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
......20 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....30 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....40 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
......50 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....60 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....70 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
......80 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....90 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....100 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....110 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
.....120 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
......130 Ocean level= 1.1494553989038812  Ocean surface fraction= 71.25
......140 Ocean level= 1.1494553989038812  Ocean surface fraction

In [15]:
#The volcanoe eroded the vastest, with a v shape forming in it's side first. This acted as a channel for the erosion
#The ring eroded slower, but both eroded eventually and the ocean level decreased.
#Because this simulation is an island simulation, coastal erosion is very prevelant, but there is no coastal erosion in this model.

In [16]:
#3. updated erosion parameters
### Physical Parameters
K = 5.0e-6 # meters^(1-2m)/yr

D = 0.001 # m^2/yr

# uplift rate
# uplift = 0.03 / 600.
uplift = 0.0

### Model parameters

# Time step
dt = d**2 / D / 8. 
#dt = d**2 / D /16. #extra small steps 
print(' dt[years] = ',dt)

#Area exponent A^m, default m=1
m=1

#gradient exponent g^n, default n=1
n=2

#erosion threshold 
theta_c = 10 

# Total simulation time
T = 1000.0 * 625.0

# total number of iterations
n_iter = int(np.round(T/dt))
print('Number of interation: ',n_iter)

# Initialize landscape 
ls = LandscapeWithOcean(NX,NY)

oceanLevelParameter=0.5  # what does this parameter do?
ls.ComputeOceanVolumeFromOceanLevelParameter(Z,NX,NY,oceanLevelParameter)

ls.pool_check(Z,NX,NY)
ls.A = np.zeros((NX,NY))

 dt[years] =  3125.0
Number of interation:  200
Minimum elevation           0.9988381863697487
Maximum elevation           1.1355137149955767
Beach level                 1.0671759506826628
Ocean volume                160.77398680034744
Percentage of ocean surface 68.66666666666667


In [17]:
#I changed the stream power k to 5.0e-6, this makes the stream power five times stronger
#I changed the diffusion rate to .0005, this makes diffusion ten times weaker
#I changed the gradient exponent n to 2, this squares the gradient and makes diffusion quadratically on the gradient
#I changed the erosion threshold theta_c to 50, this makes the erosion threshold 5 times higher meaning only very strong rivers have the power to erode.
# I changed oceanLevelParameter to 0.5, making the sea level much higher.

In [18]:
# Set-up figure
def init_figure():
    fig = plt.figure(figsize=(12.,6.))
    plt.show()
    return fig

def update_figure():
        plt.clf()
        ax1 = fig.add_subplot(121,projection='3d')

        # use equal x-y aspect with an explicit vertical exageration
        vert_exag = 4.
        ax1.set_xlim3d(0,max(NX,NY))
        ax1.set_ylim3d(0,max(NX,NY))
        ax1.set_zlim3d(0,ZMaxOrg)

        ax1.set_title('Surface Relief')

#        surf = ax1.plot_surface(X,Y,Z, cmap = cm.terrain, rstride=1, cstride=1,
#                antialiased=False,linewidth=0)

        ZPlot = np.copy(Z)
        ZPlot[ZPlot<ls.ZBeachLevel] = ls.ZBeachLevel 
        ZPlot -= ls.ZBeachLevel
        ax1.plot_surface(X,Y,ZPlot, cmap = cm.terrain, rstride=1, cstride=1,
                antialiased=False,linewidth=0)

        ax2 = fig.add_subplot(122,aspect='equal')
        ax2.set_title('Elevation')

        #im = ax2.pcolor(Z,cmap=cm.terrain)
        im = ax2.pcolor(ZPlot,cmap=cm.coolwarm)
        cs = ax2.contour(ZPlot,6,colors='k')

        # Add a color bar which maps values to colors.
        cbar = fig.colorbar(im, shrink=0.5, aspect=5)
        # Add the contour line levels to the colorbar
        cbar.add_lines(cs)

        #plt.show()
        plt.draw()
        plt.pause(0.0001)

In [19]:
with writer.saving(fig, "Homework_9_sim2.mp4", dpi=200):
    # Set up figure
    fig = init_figure()
    update_figure()
    Znew = np.copy(Z)

    for it in range(1,n_iter+1):
    
        ls.calculate_collection_area(Z,NX,NY)
        ls.A *= dx*dy
    
        for i in range(NX):
            iL = np.mod(i-1,NX) # normally i-1 but observe p.b.c.
            iR = np.mod(i+1,NX) # normally i+1 but observe p.b.c.

            for j in range(NY):
                jD = np.mod(j-1,NY) # normally j-1 but observe p.b.c.
                jU = np.mod(j+1,NY) # normally j+1 but observe p.b.c.
  
                if ls.ocean[i,j]>0:
                    Psi_z  = 0;
                    Phi_z  = 0;
                else:
                    if ls.drain[i,j]>0: #this cell is a drain
                        s1 = (Z[iR,j]  - Z[iL,j] )/(2.*dx)
                        s2 = (Z[i,jU]  - Z[i,jD] )/(2.*dy)
                        s3 = (Z[iR,jD] - Z[iL,jU])/(2. * np.sqrt( dx**2 + dy**2) )
                        s4 = (Z[iR,jU] - Z[iL,jD])/(2. * np.sqrt( dx**2 + dy**2) )
                        gradient = (np.sqrt(s1**2 + s2**2) + np.sqrt(s3**2 + s4**2))/2.
                        Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)            

                    elif ls.drainage[i,j]>0: #this cell is a drainage point (it drains a pool)
                
                        if (Z[i,j]>=Z[iR,j]) and ls.pool[iR,j]!=ls.drainage[i,j]: 
                            gradient = (Z[i,j]-Z[iR,j])/dx #pool is on my left, I drain to the right, use this gradiant
                        elif (Z[i,j]>=Z[iL,j]) and ls.pool[iL,j]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[iL,j])/dx
                        elif (Z[i,j]>=Z[i,jU]) and ls.pool[i,jU]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jU])/dy
                        elif (Z[i,j]>=Z[i,jD]) and ls.pool[i,jD]!=ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jD])/dy
                        else:
                            gradient = 0.02 # ??? This does happen (maybe when two pools merge)
                        Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)
                
                    else: #this cell is a pool, assume it has some mass diffusion but no erosion!
                        Psi_z = 0.
                
                    if (Psi_z<0):
                        Psi_z = 0. 

                    # diffusion term
                    Phi_z = D * ( (Z[iR,j] - 2.*Z[i,j] + Z[iL,j]) / dx**2 \
                                + (Z[i,jU] - 2.*Z[i,j] + Z[i,jD]) / dx**2 )
           
                Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift )*dt  

                dZdt= (Znew[i,j] - Z[i,j]) / dt
                CFL = abs(dZdt) * dt / min(dx,dy)
                if (CFL>1.0):
                    print('\nWarning: Time step of',dt,'is probably too large. Safer would be:',dt/CFL)
            
                if (Znew[i,j]<0.):
                    Znew[i,j] = 0. # yes, this does happen at the boundary when kept at zero
    
        #Znew[0,:] = 0.0 # resets front boundary to 0
        Z = np.copy(Znew)
    
        ls.pool_check(Z,NX,NY)

        if (np.mod(it,10)==0): 
            print(it,end='')
            update_figure()
            print(' Ocean level=',ls.ZBeachLevel,' Ocean surface fraction=',100*ls.AOcean/(NX*NY));
        else:
            print('.',end='')

    writer.grab_frame()
    update_figure()
    print(' Simulation finished.')


.........10 Ocean level= 1.0671441250637508  Ocean surface fraction= 71.25
......20 Ocean level= 1.0671024062285348  Ocean surface fraction= 72.66666666666667
......30 Ocean level= 1.0671024062285348  Ocean surface fraction= 73.19444444444444
.....40 Ocean level= 1.0671024062285348  Ocean surface fraction= 73.33333333333333
.....50 Ocean level= 1.0671024062285348  Ocean surface fraction= 73.55555555555556
......60 Ocean level= 1.0671024062285348  Ocean surface fraction= 73.66666666666667
.....70 Ocean level= 1.0670961042000182  Ocean surface fraction= 73.75
......80 Ocean level= 1.067095038641707  Ocean surface fraction= 73.80555555555556
......90 Ocean level= 1.067095038641707  Ocean surface fraction= 74.02777777777777
.....100 Ocean level= 1.067095038641707  Ocean surface fraction= 74.16666666666667
.....110 Ocean level= 1.067093453855058  Ocean surface fraction= 74.25
.....120 Ocean level= 1.067093453855058  Ocean surface fraction= 74.33333333333333
.....130 Ocean level= 1.067093453